# Extract text from PDFs, Word docs, and scanned images into a single Delta chunks table with the standard RAG schema

A research team has a mixed corpus they need indexed for a single RAG: three PDF policy documents, two .docx onboarding guides, and five scanned image files (.png screenshots of legacy printouts). Their previous attempt used a single `load_pdf` utility for everything, the scanned images returned empty strings, and the Word docs lost their bullet structure. You have one session to rebuild the corpus loader correctly.

The hands-on work:

- Stage ten source files (3 `.pdf`, 2 `.docx`, 5 `.png`) into a single Unity Catalog Volume under `workspace.default.labrun_genai_corpus.source_docs`.
- Implement a per-format dispatch: `pypdf` for PDFs, `python-docx` for Word, `pytesseract` for scanned images. Persist the extracted text plus `doc_uri`, `source_format`, and `ingested_at` to a Delta table `raw_docs`.
- Chunk every raw document with `RecursiveCharacterTextSplitter` (chunk_size=512, overlap=50), embed each chunk with `databricks-gte-large-en`, and write to a Delta `chunks` table with the canonical RAG schema. The table has `delta.enableChangeDataFeed = true` because Lab 5's Vector Search Delta-Sync index reads from CDF, not from a snapshot.
- Tag the schema, the volume, and the chunks table with the lab slug so the orphan scan can find them later.

The reason there are three extractors and not one: the exam tests package selection per format directly. `pypdf` parses a PDF's text layer; it returns empty strings on scanned-image PDFs because there is no text layer to parse. `pytesseract` runs OCR against the image bytes. `python-docx` walks the docx XML tree, which preserves bullets and paragraph structure that `pypdf` would flatten.

**Time.** Plan for about 65 minutes hands-on. The five OCR passes are the slowest step (50 to 90 seconds total on Free Edition serverless compute). Budget up to 105 minutes for the session window.

**Cost.** Foundation Model API embedding calls cost around one cent total: roughly 30 to 50 chunks at fractions of a cent each. Tesseract OCR runs on Free Edition serverless compute which is metered against your daily DBU quota, not against a dollar bill. A realistic session with some re-runs lands at $0.00 to $0.02. The cleanup cell at the end drops the schema, volume, and tables so nothing keeps running after you finish.

This lab maps to Databricks GenAI Engineer Associate Domain 2 (Data Preparation, ~14% provisional) and Domain 4 (Assembling and Deploying Applications, ~22% provisional).

In [ ]:
# NBVAL_SKIP
# Install labrun-checks plus the per-format extractors and the embedding
# client. Pinned versions per LAB-CREATION-BLUEPRINT.md build rules.

%pip install --quiet labrun-checks==0.7.0 databricks-sdk==0.40.0 openai==1.54.0 pypdf==4.3.1 python-docx==1.1.2 pytesseract==0.3.13 langchain-text-splitters==0.3.0 pillow==10.4.0

In [ ]:
# NBVAL_SKIP
# pytesseract is a Python binding; the underlying OCR work is done by the
# tesseract system binary. Free Edition serverless compute may or may not
# ship it pre-installed depending on the runtime image. Probe first; if the
# binary is missing, try apt-get; if apt is restricted, surface a clear
# fallback message so the student is not stuck guessing.

import shutil
import subprocess

TESSERACT_READY = False
if shutil.which("tesseract"):
    TESSERACT_READY = True
    print("tesseract binary already present on this runtime.")
else:
    try:
        result = subprocess.run(
            ["apt-get", "install", "-y", "tesseract-ocr"],
            check=False,
            capture_output=True,
            text=True,
            timeout=120,
        )
        if shutil.which("tesseract"):
            TESSERACT_READY = True
            print("tesseract installed via apt-get.")
        else:
            print("apt-get did not install tesseract. Last stderr:")
            print((result.stderr or "")[-400:])
    except Exception as exc:
        print(f"apt-get attempt failed: {exc!r}")

if not TESSERACT_READY:
    print()
    print("Fallback path: install the heavier unstructured[image] package in a")
    print("fresh cell with %pip install 'unstructured[image]==0.15.13' and replace")
    print("the extract_image() body in Task 2 with unstructured.partition.image.partition_image().")
    print("This costs ~30 extra seconds of install time but bundles its own OCR.")

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. UC identifiers (schema, table, volume) use
# snake_case under the labrun_ prefix because Unity Catalog identifiers prefer
# snake_case (hyphens force backtick-quoting everywhere downstream).

import atexit
import getpass
import io
import sys
import time
import uuid
from datetime import datetime, timezone

import pypdf
import pytesseract
from docx import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI
from PIL import Image, ImageDraw, ImageFont

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "extract-and-load-multi-format-corpus-to-delta"
LAB_TAG_KEY = "labrun_lab_slug"
LAB_TAG_VALUE = LAB_ID  # must equal cert YAML labs[3].slug exactly

CATALOG = "workspace"
PARENT_SCHEMA = "default"
LAB_SCHEMA = "labrun_genai_corpus"
LAB_SCHEMA_FQN = f"{CATALOG}.{PARENT_SCHEMA}.{LAB_SCHEMA}"
LAB_VOLUME = "source_docs"
LAB_VOLUME_FQN = f"{LAB_SCHEMA_FQN}.{LAB_VOLUME}"
LAB_VOLUME_PATH = f"/Volumes/{CATALOG}/{PARENT_SCHEMA}/{LAB_SCHEMA}/{LAB_VOLUME}"

RAW_DOCS_TABLE_FQN = f"{LAB_SCHEMA_FQN}.raw_docs"
CHUNKS_TABLE_FQN = f"{LAB_SCHEMA_FQN}.chunks"

# Ten fixture files: 3 + 2 + 5 distribution. The names are checked by the
# checkpoint via extension only; the underlying bytes are generated by the
# fixture cell so the lab is self-contained (no external download required).
FIXTURE_FILES = [
    "policy-1.pdf",
    "policy-2.pdf",
    "policy-3.pdf",
    "onboarding-1.docx",
    "onboarding-2.docx",
    "legacy-1.png",
    "legacy-2.png",
    "legacy-3.png",
    "legacy-4.png",
    "legacy-5.png",
]

EMBEDDING_MODEL = "databricks-gte-large-en"
EMBEDDING_DIM = 1024  # databricks-gte-large-en returns 1024-d vectors
CHUNK_SIZE = 512
CHUNK_OVERLAP = 50
EMBED_BATCH = 32

In [ ]:
# NBVAL_SKIP
# Register the session, validate Databricks credentials via the SDK, resolve
# the Starter SQL warehouse, and construct the OpenAI-compatible client that
# points at the Foundation Model API. Per LAB-CREATION-BLUEPRINT section 15
# this cell must succeed before the manifest cell runs.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
databricks_host = getpass.getpass("Databricks workspace URL (https://...cloud.databricks.com): ").strip()
databricks_token = getpass.getpass("Databricks personal access token (PAT): ").strip()

if not databricks_host or not databricks_token:
    print("Workspace URL and PAT are both required. Re-run this cell.")
    raise SystemExit(1)
if not databricks_host.startswith("https://"):
    databricks_host = f"https://{databricks_host}"

w = WorkspaceClient(host=databricks_host, token=databricks_token)

try:
    me = w.current_user.me()
except Exception as exc:
    print("Databricks credentials are missing or expired. CurrentUser.me() failed:")
    print(f"  {exc}")
    print("Refresh your workspace URL and PAT, then re-run this cell.")
    raise SystemExit(1)

CALLER_USER = me.user_name or (me.emails[0].value if me.emails else "unknown")
print(f"Credentials validated. Workspace user: {CALLER_USER}")

warehouses = list(w.warehouses.list())
if not warehouses:
    print("No SQL warehouses found. Free Edition ships with a Starter warehouse;")
    print("recreate it from the SQL > Warehouses page before re-running this cell.")
    raise SystemExit(1)
WAREHOUSE = warehouses[0]
WAREHOUSE_ID = WAREHOUSE.id
print(f"SQL warehouse resolved: {WAREHOUSE.name} ({WAREHOUSE_ID})")

# Foundation Model API speaks the OpenAI chat-completions and embeddings API
# shape; the only differences are the base_url and the api_key (the PAT).
openai_client = OpenAI(
    api_key=databricks_token,
    base_url=f"{databricks_host}/serving-endpoints",
)

DATABRICKS_CREDENTIALS = {
    "host": databricks_host,
    "token": databricks_token,
    "warehouse_id": WAREHOUSE_ID,
}


def run_sql(statement, warehouse_id=None, wait_seconds=180):
    """Submit a SQL statement to the warehouse and return the parsed result.

    Polls statement state up to wait_seconds, raises on FAILED or CANCELED,
    returns a dict {columns: [...], rows: [[...]]} on SUCCEEDED.
    """
    wid = warehouse_id or WAREHOUSE_ID
    resp = w.statement_execution.execute_statement(
        warehouse_id=wid,
        statement=statement,
        wait_timeout="50s",
    )
    statement_id = resp.statement_id
    deadline = time.time() + wait_seconds
    while True:
        state = resp.status.state if resp.status else None
        if state == StatementState.SUCCEEDED:
            break
        if state in (StatementState.FAILED, StatementState.CANCELED, StatementState.CLOSED):
            err = resp.status.error.message if (resp.status and resp.status.error) else "no error message"
            raise RuntimeError(f"SQL failed ({state}): {err}\n  Statement: {statement}")
        if time.time() > deadline:
            raise TimeoutError(
                f"SQL did not finish in {wait_seconds}s. Last state: {state}. "
                f"The Starter warehouse may still be waking up; re-run the cell."
            )
        time.sleep(2)
        resp = w.statement_execution.get_statement(statement_id)

    columns = []
    if resp.manifest and resp.manifest.schema and resp.manifest.schema.columns:
        columns = [c.name for c in resp.manifest.schema.columns]
    rows = []
    if resp.result and resp.result.data_array:
        rows = list(resp.result.data_array)
    return {"columns": columns, "rows": rows}


register(session_token=session_token, lab_id=LAB_ID, credentials=DATABRICKS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan. Reverse-creation
# order. No critical resources on Free Edition (no hourly billing), so the
# canonical summary always reports zero critical destructions. Per
# RESOURCE-SAFETY-SPEC Rule 4 the orphan scan blocks execution if any
# tagged resources from a prior session exist.


CLEANUP_MANIFEST = [
    CleanupResource(
        type="uc_managed_table",
        id=CHUNKS_TABLE_FQN,
        region="databricks",
        cli_delete_command=f'databricks sql -e "DROP TABLE IF EXISTS {CHUNKS_TABLE_FQN}"',
    ),
    CleanupResource(
        type="uc_managed_table",
        id=RAW_DOCS_TABLE_FQN,
        region="databricks",
        cli_delete_command=f'databricks sql -e "DROP TABLE IF EXISTS {RAW_DOCS_TABLE_FQN}"',
    ),
    CleanupResource(
        type="uc_volume_contents",
        id=LAB_VOLUME_FQN,
        region="databricks",
        cli_delete_command=f"databricks fs rm -r dbfs:{LAB_VOLUME_PATH}/",
    ),
    CleanupResource(
        type="uc_volume",
        id=LAB_VOLUME_FQN,
        region="databricks",
        cli_delete_command=f'databricks sql -e "DROP VOLUME IF EXISTS {LAB_VOLUME_FQN}"',
    ),
    CleanupResource(
        type="uc_schema",
        id=LAB_SCHEMA_FQN,
        region="databricks",
        cli_delete_command=f'databricks sql -e "DROP SCHEMA IF EXISTS {LAB_SCHEMA_FQN} CASCADE"',
    ),
]


class DatabricksCleanupAdapter:
    """Inline adapter implementing the labrun-checks CleanupAdapter protocol
    against Databricks Unity Catalog via the SQL Statement Execution API.
    Targets uc_managed_table, uc_volume, uc_volume_contents, and uc_schema
    resources."""

    def delete_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "uc_managed_table":
            run_sql(f"DROP TABLE IF EXISTS {rid}")
        elif rtype == "uc_volume":
            run_sql(f"DROP VOLUME IF EXISTS {rid}")
        elif rtype == "uc_volume_contents":
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                listing = list(w.files.list_directory_contents(volume_path))
            except Exception:
                return
            for entry in listing:
                try:
                    if entry.is_directory:
                        w.files.delete_directory(entry.path)
                    else:
                        w.files.delete(entry.path)
                except Exception:
                    pass
        elif rtype == "uc_schema":
            run_sql(f"DROP SCHEMA IF EXISTS {rid} CASCADE")
        else:
            raise ValueError(f"DatabricksCleanupAdapter: unknown resource type {rtype!r}")

    def describe_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "uc_managed_table":
            catalog, schema, table = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.tables "
                f"WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' "
                f"AND table_name = '{table}'"
            )
            result = run_sql(sql)
            return len(result["rows"]) > 0
        if rtype == "uc_volume":
            catalog, schema, volume = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.volumes "
                f"WHERE volume_catalog = '{catalog}' AND volume_schema = '{schema}' "
                f"AND volume_name = '{volume}'"
            )
            result = run_sql(sql)
            return len(result["rows"]) > 0
        if rtype == "uc_volume_contents":
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                listing = list(w.files.list_directory_contents(volume_path))
                return len(listing) > 0
            except Exception:
                return False
        if rtype == "uc_schema":
            catalog, schema = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.schemata "
                f"WHERE catalog_name = '{catalog}' AND schema_name = '{schema}'"
            )
            result = run_sql(sql)
            return len(result["rows"]) > 0
        return False

    def tag_scan(self, credentials, lab_slug, region):
        """Return list of Unity Catalog identifiers tagged with the lab slug."""
        arns = []
        queries = [
            (
                "SELECT catalog_name, schema_name FROM system.information_schema.schema_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}",
            ),
            (
                "SELECT catalog_name, schema_name, table_name FROM system.information_schema.table_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
            (
                "SELECT catalog_name, schema_name, volume_name FROM system.information_schema.volume_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
        ]
        for sql, fmt in queries:
            try:
                result = run_sql(sql)
            except Exception:
                continue
            for row in result["rows"]:
                arns.append(fmt(row))
        return arns


CLEANUP_ADAPTER = DatabricksCleanupAdapter()


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans():
    return CLEANUP_ADAPTER.tag_scan(DATABRICKS_CREDENTIALS, LAB_TAG_VALUE, "databricks")


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing UC objects tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can produce duplicate resources.")
    print("Run the cleanup cell at the bottom of this notebook first, or")
    print("drop them manually with the SQL commands shown by the cleanup")
    print("cell, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Create the schema, the source_docs volume, and stage ten fixture files

Three statements get the storage scaffolding in place:

1. `CREATE SCHEMA IF NOT EXISTS workspace.default.labrun_genai_corpus` so the lab is re-runnable.
2. `CREATE VOLUME IF NOT EXISTS workspace.default.labrun_genai_corpus.source_docs` so the ten fixture files have a UC-governed home (volumes are the UC equivalent of object-store mounts; they have grants and they tag).
3. After the schema and volume exist, run the fixture-generator cell below to write ten files into the volume: three small PDFs, two Word docs, five PNGs. The fixture-generator builds the bytes in-process and uploads via the Files API. No external download. The PNGs are deliberately small images rendered with PIL so `pytesseract` has something legible to OCR.

The checkpoint reads the volume listing and asserts the count plus the extension distribution (3 pdf, 2 docx, 5 png). The bytes do not need to match a specific corpus; the OCR'd content lands in raw_docs in Task 2 and the checkpoint there asserts on length.

In [ ]:
# NBVAL_SKIP
# Task 1: Create the schema, create the volume, and apply the lab tag to
# both. The CREATEs are IF NOT EXISTS so a kernel restart mid-lab does
# not blow up. The fixture-generator cell below this one handles the
# ten file uploads.

# YOUR CODE: Run CREATE SCHEMA IF NOT EXISTS for LAB_SCHEMA_FQN via run_sql().

# YOUR CODE: Run CREATE VOLUME IF NOT EXISTS for LAB_VOLUME_FQN via run_sql().

# YOUR CODE: Tag the schema with ALTER SCHEMA ... SET TAGS
# ('labrun_lab_slug' = LAB_TAG_VALUE) via run_sql().

# YOUR CODE: Tag the volume with ALTER VOLUME ... SET TAGS
# ('labrun_lab_slug' = LAB_TAG_VALUE) via run_sql().

print(f"Schema and volume ready: {LAB_SCHEMA_FQN}, {LAB_VOLUME_FQN}")

In [ ]:
# NBVAL_SKIP
# Fixture generator. Writes ten files into LAB_VOLUME_PATH using only the
# libraries already imported (pypdf, python-docx, PIL). No external download.
# The PNG content is rendered text so pytesseract has something to OCR.

POLICY_TEXTS = [
    (
        "Acme Corporation Data Retention Policy\n\n"
        "Customer interaction records are retained for seven years from the date of last contact. "
        "Financial transaction records are retained for ten years from the close of the fiscal year. "
        "Application logs containing personal identifiers are retained for ninety days. "
        "Deletion requests under GDPR Article 17 are honored within thirty days unless a legal "
        "hold supersedes. The Data Protection Officer is the escalation contact for any conflict "
        "between this policy and a regulatory request."
    ),
    (
        "Acme Incident Response Policy\n\n"
        "A security incident is defined as any event that compromises the confidentiality, integrity, "
        "or availability of a customer-facing system. The on-call engineer files an INC ticket within "
        "fifteen minutes of detection. The security lead is paged for any incident at severity P0 or P1. "
        "Post-incident review is mandatory within five business days and the writeup is published to "
        "the engineering wiki within ten business days. Severity downgrades require sign-off from the "
        "incident commander."
    ),
    (
        "Acme Acceptable Use Policy\n\n"
        "Workstations issued by Acme are for business use. Personal use is permitted in moderation "
        "subject to manager judgment. Installation of unapproved software is prohibited. Storing "
        "customer data on a personal device is prohibited under any circumstance. Violations are "
        "reviewed by Security and Legal and may result in revocation of access. Reporting a violation "
        "in good faith does not result in retaliation; this is the cornerstone of the program."
    ),
]

ONBOARDING_TEXTS = [
    (
        "Engineering Onboarding Week One",
        [
            "Day 1: laptop pickup, badge access, and intro session with your manager.",
            "Day 2: development environment setup and your first pull request against the docs repo.",
            "Day 3: pair-program on a real ticket with your assigned onboarding buddy.",
            "Day 4: ship a small fix to production behind a feature flag.",
            "Day 5: retro with your manager. Set goals for week two.",
        ],
    ),
    (
        "Sales Onboarding Week One",
        [
            "Monday: CRM training plus access to the sandbox tenant.",
            "Tuesday: shadow a discovery call and write up your notes by EOD.",
            "Wednesday: product deep-dive with a solutions engineer.",
            "Thursday: roleplay an objection-handling drill with your enablement partner.",
            "Friday: pipeline review with your manager and a plan for week two outreach.",
        ],
    ),
]

LEGACY_PRINT_TEXTS = [
    "INVOICE 2017 0423\nCustomer Atlas Industrial\nAmount Due 4250.00 USD",
    "PURCHASE ORDER 9981\nVendor Globex\nLine Item Widget Assembly Qty 200",
    "EXPENSE REPORT Q3 2018\nEmployee J Romero\nTotal Reimbursable 1820.55",
    "MEETING MINUTES 2016 0712\nProject Sentinel\nAction Items Carry Over 3",
    "CONTRACT ADDENDUM\nEffective Date 2015 0901\nTerm Extension Six Months",
]


def _write_pdf(path, text):
    """Build a minimal text-only PDF and upload it via the Files API.

    Emits a valid single-page PDF whose text layer extracts cleanly with
    pypdf.PdfReader(). No reportlab dependency."""
    lines = text.split("\n")
    parts = []
    for i, line in enumerate(lines):
        safe = line.replace("\\", "\\\\").replace("(", "\\(").replace(")", "\\)")
        if i == 0:
            parts.append(f"({safe}) Tj")
        else:
            parts.append(f"T* ({safe}) Tj")
    body = "\n".join(parts)
    stream = f"BT /F1 11 Tf 50 770 Td 14 TL\n{body}\nET".encode("utf-8")
    objs = [
        b"1 0 obj << /Type /Catalog /Pages 2 0 R >> endobj",
        b"2 0 obj << /Type /Pages /Kids [3 0 R] /Count 1 >> endobj",
        (
            b"3 0 obj << /Type /Page /Parent 2 0 R /MediaBox [0 0 612 792] "
            b"/Resources << /Font << /F1 5 0 R >> >> /Contents 4 0 R >> endobj"
        ),
        b"4 0 obj << /Length " + str(len(stream)).encode() + b" >> stream\n"
        + stream + b"\nendstream endobj",
        b"5 0 obj << /Type /Font /Subtype /Type1 /BaseFont /Helvetica >> endobj",
    ]
    pdf = b"%PDF-1.4\n"
    offsets = []
    for obj in objs:
        offsets.append(len(pdf))
        pdf += obj + b"\n"
    xref_offset = len(pdf)
    pdf += f"xref\n0 {len(objs) + 1}\n0000000000 65535 f \n".encode()
    for off in offsets:
        pdf += f"{off:010d} 00000 n \n".encode()
    pdf += f"trailer << /Size {len(objs) + 1} /Root 1 0 R >>\nstartxref\n{xref_offset}\n%%EOF\n".encode()
    w.files.upload(path, io.BytesIO(pdf), overwrite=True)


def _write_docx(path, title, bullets):
    doc = Document()
    doc.add_heading(title, level=1)
    for b in bullets:
        doc.add_paragraph(b, style="List Bullet")
    buf = io.BytesIO()
    doc.save(buf)
    buf.seek(0)
    w.files.upload(path, buf, overwrite=True)


def _write_png(path, text):
    img = Image.new("RGB", (640, 200), color="white")
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 22)
    except Exception:
        font = ImageFont.load_default()
    y = 20
    for line in text.split("\n"):
        draw.text((20, y), line, fill="black", font=font)
        y += 32
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    buf.seek(0)
    w.files.upload(path, buf, overwrite=True)


for i, body in enumerate(POLICY_TEXTS, start=1):
    _write_pdf(f"{LAB_VOLUME_PATH}/policy-{i}.pdf", body)
for i, (title, bullets) in enumerate(ONBOARDING_TEXTS, start=1):
    _write_docx(f"{LAB_VOLUME_PATH}/onboarding-{i}.docx", title, bullets)
for i, body in enumerate(LEGACY_PRINT_TEXTS, start=1):
    _write_png(f"{LAB_VOLUME_PATH}/legacy-{i}.png", body)

listing = list(w.files.list_directory_contents(LAB_VOLUME_PATH))
print(f"Files staged in {LAB_VOLUME_PATH}: {len(listing)}")
for entry in sorted(listing, key=lambda e: e.path):
    print("  -", entry.path)

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Ten files staged in the volume with the expected
# distribution (3 pdf, 2 docx, 5 png or jpeg).


def checkpoint_1(session):
    try:
        listing = list(w.files.list_directory_contents(LAB_VOLUME_PATH))
        if len(listing) != 10:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{LAB_VOLUME_PATH} has {len(listing)} files; expected exactly 10. "
                    f"Re-run the fixture-generator cell."
                ),
            )
        by_ext = {"pdf": 0, "docx": 0, "png": 0, "jpeg": 0}
        for entry in listing:
            name = entry.path.rsplit("/", 1)[-1].lower()
            ext = name.rsplit(".", 1)[-1] if "." in name else ""
            if ext in by_ext:
                by_ext[ext] += 1
        if by_ext["pdf"] != 3:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Volume has {by_ext['pdf']} .pdf files; expected 3. "
                    f"Check the fixture-generator output."
                ),
            )
        if by_ext["docx"] != 2:
            return CheckpointResult(
                status="fail",
                error_reason=f"Volume has {by_ext['docx']} .docx files; expected 2.",
            )
        image_count = by_ext["png"] + by_ext["jpeg"]
        if image_count != 5:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Volume has {image_count} image files (.png or .jpeg); expected 5."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint message tells you which extension count is off. If the count is zero everywhere, the fixture-generator cell did not run; both the schema and the volume must exist before the fixture cell can upload anything. If the schema exists but the volume does not, the fixture cell will throw on its first upload.

</details>

<details><summary>Hint 2 (direction)</summary>

Two CREATE statements plus two ALTER ... SET TAGS statements, all run through `run_sql()`. The constants `LAB_SCHEMA_FQN`, `LAB_VOLUME_FQN`, `LAB_TAG_KEY`, and `LAB_TAG_VALUE` are already in scope. After both objects exist and are tagged, the fixture-generator cell directly below the task cell handles the ten uploads; run it next.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
run_sql(f"CREATE SCHEMA IF NOT EXISTS {LAB_SCHEMA_FQN}")
run_sql(f"CREATE VOLUME IF NOT EXISTS {LAB_VOLUME_FQN}")
run_sql(
    f"ALTER SCHEMA {LAB_SCHEMA_FQN} "
    f"SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')"
)
run_sql(
    f"ALTER VOLUME {LAB_VOLUME_FQN} "
    f"SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')"
)
```

After this passes, run the fixture-generator cell to upload the 10 files.

</details>

**Wallet check.** Still at $0.00. Four metastore writes plus ten tiny file uploads (the largest fixture is under 20 KB). Zero embedding calls, zero OCR runs. The daily compute quota has barely moved.

## Task 2: Per-format extraction into a raw_docs Delta table

Three extractors, one dispatch loop:

- `extract_pdf(path)` uses `pypdf.PdfReader(path)` and joins `page.extract_text()` across every page. This works on text-PDFs (which is what the fixture pack ships) and returns empty strings on scanned-image PDFs (which is why the next extractor exists).
- `extract_docx(path)` uses `python-docx`'s `Document(path)` and joins `paragraph.text` across every paragraph. Bullets and headings come through as text; tables would need a separate walk.
- `extract_image(path)` uses `pytesseract.image_to_string(Image.open(path))`. This is the OCR path. It runs on every `.png` and `.jpeg`.

The dispatch loop iterates the volume listing, picks the extractor by extension, and inserts one row per document into `raw_docs` with columns `(doc_uri, source_format, raw_text, ingested_at)`. The checkpoint asserts ten rows, the per-format distribution (3 pdf, 2 docx, 5 png), and `LENGTH(raw_text) > 100` on every row to confirm the extractors actually pulled real content.

The five OCR passes are the slowest step in the entire lab. Expect 10 to 15 seconds per image on Free Edition serverless compute.

In [ ]:
# NBVAL_SKIP
# Task 2: Implement three extractors plus the dispatch loop, then write the
# resulting rows to raw_docs. The CREATE TABLE for raw_docs is shown here
# because the schema is fixed; the extractor implementations are yours.

run_sql(
    f"CREATE TABLE IF NOT EXISTS {RAW_DOCS_TABLE_FQN} ("
    f"  doc_uri STRING NOT NULL, "
    f"  source_format STRING NOT NULL, "
    f"  raw_text STRING NOT NULL, "
    f"  ingested_at TIMESTAMP NOT NULL"
    f") USING DELTA"
)

# Truncate so re-runs are idempotent. The CREATE is IF NOT EXISTS; the
# truncate handles the second-run case.
run_sql(f"TRUNCATE TABLE {RAW_DOCS_TABLE_FQN}")


def extract_pdf(path):
    # YOUR CODE: open the PDF at `path` with pypdf.PdfReader and return the
    # concatenated text from every page.
    return ""


def extract_docx(path):
    # YOUR CODE: open the docx at `path` with python-docx's Document() and
    # return the concatenated text of every paragraph.
    return ""


def extract_image(path):
    # YOUR CODE: open the image at `path` with PIL.Image.open and OCR it via
    # pytesseract.image_to_string().
    return ""


# Dispatch loop. Reads the volume, picks an extractor by extension, inserts
# one row per document. raw_text gets escaped for the SQL string literal.
listing = list(w.files.list_directory_contents(LAB_VOLUME_PATH))
ingested_at = datetime.now(timezone.utc).isoformat()
inserted = 0
for entry in sorted(listing, key=lambda e: e.path):
    if entry.is_directory:
        continue
    path = entry.path
    name = path.rsplit("/", 1)[-1].lower()
    ext = name.rsplit(".", 1)[-1]
    if ext == "pdf":
        source_format = "pdf"
        text = extract_pdf(path)
    elif ext == "docx":
        source_format = "docx"
        text = extract_docx(path)
    elif ext in ("png", "jpeg", "jpg"):
        source_format = "png" if ext == "png" else "jpeg"
        text = extract_image(path)
    else:
        continue
    safe_text = text.replace("\\", "\\\\").replace("'", "''")
    run_sql(
        f"INSERT INTO {RAW_DOCS_TABLE_FQN} VALUES ("
        f"  '{path}', '{source_format}', '{safe_text}', TIMESTAMP'{ingested_at}'"
        f")"
    )
    inserted += 1

print(f"Inserted {inserted} rows into {RAW_DOCS_TABLE_FQN}")

result = run_sql(
    f"SELECT source_format, count(*) FROM {RAW_DOCS_TABLE_FQN} GROUP BY source_format"
)
for row in result["rows"]:
    print(f"  {row[0]}: {row[1]} rows")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: raw_docs has exactly 10 rows with the expected per-format
# distribution and non-trivial raw_text length on every row.


def checkpoint_2(session):
    try:
        count_result = run_sql(f"SELECT count(*) FROM {RAW_DOCS_TABLE_FQN}")
        row_count = int(count_result["rows"][0][0]) if count_result["rows"] else 0
        if row_count != 10:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{RAW_DOCS_TABLE_FQN} has {row_count} rows; expected exactly 10. "
                    f"Truncate and re-run the dispatch loop."
                ),
            )
        fmt_result = run_sql(
            f"SELECT source_format, count(*) FROM {RAW_DOCS_TABLE_FQN} "
            f"GROUP BY source_format ORDER BY source_format"
        )
        counts = {row[0]: int(row[1]) for row in fmt_result["rows"]}
        if counts.get("pdf", 0) != 3:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"pdf rows: {counts.get('pdf', 0)}; expected 3. "
                    f"extract_pdf() may be returning empty strings."
                ),
            )
        if counts.get("docx", 0) != 2:
            return CheckpointResult(
                status="fail",
                error_reason=f"docx rows: {counts.get('docx', 0)}; expected 2.",
            )
        image_rows = counts.get("png", 0) + counts.get("jpeg", 0)
        if image_rows != 5:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"image rows (png + jpeg): {image_rows}; expected 5. "
                    f"extract_image() may be returning empty strings (check that "
                    f"tesseract is installed)."
                ),
            )
        short_result = run_sql(
            f"SELECT doc_uri, LENGTH(raw_text) AS n FROM {RAW_DOCS_TABLE_FQN} "
            f"WHERE LENGTH(raw_text) <= 100"
        )
        if short_result["rows"]:
            offenders = ", ".join(
                f"{row[0]} (len={row[1]})" for row in short_result["rows"][:3]
            )
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Some rows have raw_text shorter than 100 chars: {offenders}. "
                    f"The extractor for that format is returning empty or truncated text."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names the failing format (pdf, docx, or image). If pdf rows have short text, `extract_pdf()` is probably returning the wrong attribute. If image rows have short text, the tesseract binary is missing (re-check the install probe in the second cell). If docx rows have short text, `python-docx`'s `Document().paragraphs` returns Paragraph objects, not strings; you need `.text` on each.

</details>

<details><summary>Hint 2 (direction)</summary>

Three one-liners. `extract_pdf`: build a `pypdf.PdfReader(path)`, then join `page.extract_text()` across `reader.pages`. `extract_docx`: build a `Document(path)`, then join `p.text` across `doc.paragraphs`. `extract_image`: open with `Image.open(path)`, then call `pytesseract.image_to_string()`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
def extract_pdf(path):
    reader = pypdf.PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)


def extract_docx(path):
    doc = Document(path)
    return "\n".join(p.text for p in doc.paragraphs if p.text)


def extract_image(path):
    return pytesseract.image_to_string(Image.open(path))
```

UC Volume paths are readable directly by `open()` and by libraries that accept a filesystem path. If your runtime returns binary streams instead of paths, use `w.files.download(path).contents` to pull bytes then wrap in `io.BytesIO` before passing to the extractor.

</details>

**Wallet check.** Roughly $0.00 still. Five OCR passes took 50 to 90 seconds of serverless compute against your daily DBU quota (the slowest single step of this lab, but free). Three PDF parses and two docx parses were instantaneous. Zero embedding calls so far. Your morning coffee remains the most expensive thing you have purchased today.

## Task 3: Chunk, embed, and write to the chunks table

This is where the corpus becomes RAG-ready. Three pieces:

1. **Chunker.** `RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50, separators=["\n\n", "\n", ". ", " "])`. This is the Lab 3 winning chunker for documents with mixed paragraph and sentence structure. Apply it to every `raw_docs.raw_text`.
2. **Embedder.** `openai_client.embeddings.create(model="databricks-gte-large-en", input=batch_of_up_to_32_chunks)`. The embedding API accepts up to 64 inputs per call; the lab batches in groups of 32 to stay safely under the limit. Each returned embedding is a 1024-dimensional float vector.
3. **Persist.** Write to `workspace.default.labrun_genai_corpus.chunks` with `(chunk_id, doc_uri, chunk_index, chunk_text, embedding, source_format, ingested_at)`. The table is created with `delta.enableChangeDataFeed = true` because Lab 5's Vector Search Delta-Sync index reads from CDF, not from a snapshot scan.

The checkpoint asserts at least 30 rows total (a 10-document corpus with average 3 to 5 chunks per doc lands comfortably there), `size(embedding) = 1024` on every row, `chunk_text` non-empty on every row, `source_format` in (pdf, docx, png, jpeg), and `chunk_index >= 0`.

In [ ]:
# NBVAL_SKIP
# Task 3: Chunk every raw_docs row, embed each chunk via the Foundation
# Model API, persist to the chunks table. The CREATE TABLE here pins
# delta.enableChangeDataFeed = true because Lab 5 depends on it.

run_sql(
    f"CREATE TABLE IF NOT EXISTS {CHUNKS_TABLE_FQN} ("
    f"  chunk_id STRING NOT NULL, "
    f"  doc_uri STRING NOT NULL, "
    f"  chunk_index INT NOT NULL, "
    f"  chunk_text STRING NOT NULL, "
    f"  embedding ARRAY<FLOAT> NOT NULL, "
    f"  source_format STRING NOT NULL, "
    f"  ingested_at TIMESTAMP NOT NULL"
    f") USING DELTA "
    f"TBLPROPERTIES (delta.enableChangeDataFeed = true)"
)
run_sql(f"TRUNCATE TABLE {CHUNKS_TABLE_FQN}")

# YOUR CODE: Build a RecursiveCharacterTextSplitter named `splitter` with
# chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, and separators
# ["\n\n", "\n", ". ", " "].
splitter = None

# Read every raw_docs row.
raw_result = run_sql(
    f"SELECT doc_uri, source_format, raw_text FROM {RAW_DOCS_TABLE_FQN}"
)

# Build the (chunk_id, doc_uri, chunk_index, chunk_text, source_format)
# tuple list first so we can batch-embed efficiently.
pending = []
for doc_uri, source_format, raw_text in raw_result["rows"]:
    # YOUR CODE: split raw_text into `chunks_for_doc` using splitter.split_text().
    chunks_for_doc = []
    for idx, chunk_text in enumerate(chunks_for_doc):
        chunk_id = str(uuid.uuid4())
        pending.append((chunk_id, doc_uri, idx, chunk_text, source_format))

print(f"Total chunks pending embedding: {len(pending)}")


def embed_batch(texts):
    """Call the Foundation Model API embedding endpoint for a batch of up to
    EMBED_BATCH strings and return a list of float vectors."""
    # YOUR CODE: call openai_client.embeddings.create(model=EMBEDDING_MODEL,
    # input=texts) and return [item.embedding for item in resp.data].
    return []


ingested_at = datetime.now(timezone.utc).isoformat()
written = 0
for batch_start in range(0, len(pending), EMBED_BATCH):
    batch = pending[batch_start : batch_start + EMBED_BATCH]
    texts = [row[3] for row in batch]
    vectors = embed_batch(texts)
    if len(vectors) != len(batch):
        raise RuntimeError(
            f"Embedding response size mismatch: expected {len(batch)}, got {len(vectors)}"
        )
    values_clauses = []
    for (chunk_id, doc_uri, idx, chunk_text, source_format), vec in zip(batch, vectors):
        safe_text = chunk_text.replace("\\", "\\\\").replace("'", "''")
        vec_literal = "ARRAY(" + ", ".join(f"CAST({v} AS FLOAT)" for v in vec) + ")"
        values_clauses.append(
            f"('{chunk_id}', '{doc_uri}', {idx}, '{safe_text}', {vec_literal}, "
            f"'{source_format}', TIMESTAMP'{ingested_at}')"
        )
    insert_sql = (
        f"INSERT INTO {CHUNKS_TABLE_FQN} "
        f"(chunk_id, doc_uri, chunk_index, chunk_text, embedding, source_format, ingested_at) "
        f"VALUES " + ", ".join(values_clauses)
    )
    run_sql(insert_sql)
    written += len(batch)
    print(f"  wrote batch of {len(batch)} (running total: {written})")

count_result = run_sql(f"SELECT count(*) FROM {CHUNKS_TABLE_FQN}")
print(f"Total rows in {CHUNKS_TABLE_FQN}: {count_result['rows'][0][0]}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: chunks table has at least 30 rows, every embedding is
# 1024-dim, every chunk_text is non-empty, every source_format is one of
# the four allowed values, every chunk_index is non-negative.


def checkpoint_3(session):
    try:
        count_result = run_sql(f"SELECT count(*) FROM {CHUNKS_TABLE_FQN}")
        row_count = int(count_result["rows"][0][0]) if count_result["rows"] else 0
        if row_count < 30:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{CHUNKS_TABLE_FQN} has {row_count} rows; expected at least 30. "
                    f"Check that the chunker produced multiple chunks per document."
                ),
            )

        dim_result = run_sql(
            f"SELECT count(*) FROM {CHUNKS_TABLE_FQN} WHERE size(embedding) != {EMBEDDING_DIM}"
        )
        bad_dim = int(dim_result["rows"][0][0])
        if bad_dim > 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{bad_dim} rows have embedding size != {EMBEDDING_DIM}. "
                    f"databricks-gte-large-en returns 1024-d vectors; confirm "
                    f"EMBEDDING_MODEL is set correctly."
                ),
            )

        empty_result = run_sql(
            f"SELECT count(*) FROM {CHUNKS_TABLE_FQN} WHERE chunk_text IS NULL OR LENGTH(chunk_text) = 0"
        )
        empty_chunks = int(empty_result["rows"][0][0])
        if empty_chunks > 0:
            return CheckpointResult(
                status="fail",
                error_reason=f"{empty_chunks} rows have empty chunk_text.",
            )

        fmt_result = run_sql(
            f"SELECT count(*) FROM {CHUNKS_TABLE_FQN} "
            f"WHERE source_format NOT IN ('pdf', 'docx', 'png', 'jpeg')"
        )
        bad_fmt = int(fmt_result["rows"][0][0])
        if bad_fmt > 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{bad_fmt} rows have a source_format outside (pdf, docx, png, jpeg)."
                ),
            )

        idx_result = run_sql(
            f"SELECT count(*) FROM {CHUNKS_TABLE_FQN} WHERE chunk_index < 0"
        )
        bad_idx = int(idx_result["rows"][0][0])
        if bad_idx > 0:
            return CheckpointResult(
                status="fail",
                error_reason=f"{bad_idx} rows have chunk_index < 0.",
            )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names which constraint failed: row count too low (the chunker is not splitting), embedding size wrong (the model name is off), chunk_text empty (the splitter returned empty strings, probably because raw_text was empty for that document), bad source_format (the dispatch loop set the wrong value), or chunk_index negative (an enumeration started at the wrong value).

</details>

<details><summary>Hint 2 (direction)</summary>

`splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, separators=["\n\n", "\n", ". ", " "])`. Then `chunks_for_doc = splitter.split_text(raw_text)`. The embedding call is `resp = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=texts)` and the vectors are `[item.embedding for item in resp.data]`. The OpenAI client returns a Pydantic model; iterate `resp.data`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " "],
)

# inside the loop over raw_result["rows"]:
chunks_for_doc = splitter.split_text(raw_text)


def embed_batch(texts):
    resp = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in resp.data]
```

</details>

**Wallet check.** About one cent. The 30 to 50 chunks went through the embedding endpoint in two batches of up to 32 inputs each. Foundation Model API embedding calls bill against the workspace token quota at fractions of a cent per call. Two batched calls cost roughly $0.01 total. Five OCR passes earlier cost zero dollars (just serverless compute against the daily quota).

## Task 4: Tag the chunks table and confirm CDF is on

The schema and the volume already carry the lab tag from Task 1. The `chunks` table and the `raw_docs` table do not yet (tags do not propagate from schema down to tables in Unity Catalog). Two pieces here:

1. Apply `ALTER TABLE ... SET TAGS ('labrun_lab_slug' = '...')` to both `raw_docs` and `chunks`. The orphan scan and the cleanup tag audit both rely on this.
2. Confirm `chunks` was created with `delta.enableChangeDataFeed = true`. The `CREATE TABLE` in Task 3 pinned this in TBLPROPERTIES, but Lab 5's Vector Search Delta-Sync index will silently fail to sync new rows if CDF is off. If you re-ran Task 3 with the CDF clause removed by accident, fix it now with `ALTER TABLE ... SET TBLPROPERTIES (delta.enableChangeDataFeed = true)`.

The checkpoint queries `system.information_schema.schema_tags`, `volume_tags`, and `table_tags` for the lab slug on each of the three entities, then parses `DESCRIBE TABLE EXTENDED chunks` for the CDF flag.

In [ ]:
# NBVAL_SKIP
# Task 4: Apply the lab tag to raw_docs and chunks (the schema and volume
# were tagged in Task 1). Then confirm chunks has CDF enabled.

# YOUR CODE: Run ALTER TABLE RAW_DOCS_TABLE_FQN SET TAGS
# ('labrun_lab_slug' = LAB_TAG_VALUE) via run_sql().

# YOUR CODE: Run ALTER TABLE CHUNKS_TABLE_FQN SET TAGS
# ('labrun_lab_slug' = LAB_TAG_VALUE) via run_sql().

# Defensive re-apply of CDF in case a re-run of Task 3 dropped the
# TBLPROPERTIES. ALTER TABLE ... SET TBLPROPERTIES is idempotent.
run_sql(
    f"ALTER TABLE {CHUNKS_TABLE_FQN} "
    f"SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
)

print("Tags applied to raw_docs and chunks. CDF re-confirmed on chunks.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: the schema, the volume, and the chunks table all carry the
# lab tag; chunks has delta.enableChangeDataFeed = true.


def checkpoint_4(session):
    try:
        schema_tag = run_sql(
            "SELECT tag_value FROM system.information_schema.schema_tags "
            f"WHERE catalog_name = '{CATALOG}' AND schema_name = '{LAB_SCHEMA}' "
            f"AND tag_name = '{LAB_TAG_KEY}'"
        )
        if not any(row[0] == LAB_TAG_VALUE for row in schema_tag["rows"]):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Schema {LAB_SCHEMA_FQN} is missing tag {LAB_TAG_KEY}={LAB_TAG_VALUE}. "
                    f"Run ALTER SCHEMA {LAB_SCHEMA_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')."
                ),
            )

        volume_tag = run_sql(
            "SELECT tag_value FROM system.information_schema.volume_tags "
            f"WHERE catalog_name = '{CATALOG}' AND schema_name = '{LAB_SCHEMA}' "
            f"AND volume_name = '{LAB_VOLUME}' AND tag_name = '{LAB_TAG_KEY}'"
        )
        if not any(row[0] == LAB_TAG_VALUE for row in volume_tag["rows"]):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Volume {LAB_VOLUME_FQN} is missing tag {LAB_TAG_KEY}={LAB_TAG_VALUE}. "
                    f"Run ALTER VOLUME {LAB_VOLUME_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')."
                ),
            )

        table_tag = run_sql(
            "SELECT tag_value FROM system.information_schema.table_tags "
            f"WHERE catalog_name = '{CATALOG}' AND schema_name = '{LAB_SCHEMA}' "
            f"AND table_name = 'chunks' AND tag_name = '{LAB_TAG_KEY}'"
        )
        if not any(row[0] == LAB_TAG_VALUE for row in table_tag["rows"]):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Table {CHUNKS_TABLE_FQN} is missing tag {LAB_TAG_KEY}={LAB_TAG_VALUE}. "
                    f"Run ALTER TABLE {CHUNKS_TABLE_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')."
                ),
            )

        desc = run_sql(f"DESCRIBE TABLE EXTENDED {CHUNKS_TABLE_FQN}")
        cdf_on = False
        for row in desc["rows"]:
            # DESCRIBE TABLE EXTENDED returns rows of shape
            # [col_name, data_type, comment]. TBLPROPERTIES lands in a row
            # whose col_name is 'Table Properties' and whose data_type holds
            # the bracketed key=value list.
            if not row:
                continue
            line = " ".join(str(c) for c in row if c is not None).lower()
            if "delta.enablechangedatafeed" in line and "true" in line:
                cdf_on = True
                break
        if not cdf_on:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{CHUNKS_TABLE_FQN} does not have delta.enableChangeDataFeed = true. "
                    f"Run ALTER TABLE {CHUNKS_TABLE_FQN} SET TBLPROPERTIES "
                    f"(delta.enableChangeDataFeed = true). Lab 5's Vector Search "
                    f"Delta-Sync index reads from CDF, not from a snapshot."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names which entity is missing the tag (schema, volume, or table) or whether CDF is off. Tags do not propagate down the Unity Catalog hierarchy; each entity needs its own `SET TAGS` call. The schema and volume tags were set in Task 1; the table tags are the ones to set here.

</details>

<details><summary>Hint 2 (direction)</summary>

Two `ALTER TABLE ... SET TAGS` statements for raw_docs and chunks. The CDF reset (`SET TBLPROPERTIES`) is already in the cell. If the checkpoint still complains about CDF, the issue is the CREATE TABLE statement in Task 3 dropped the TBLPROPERTIES clause; re-running the SET TBLPROPERTIES line in this cell fixes it.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4:

```python
run_sql(
    f"ALTER TABLE {RAW_DOCS_TABLE_FQN} "
    f"SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')"
)
run_sql(
    f"ALTER TABLE {CHUNKS_TABLE_FQN} "
    f"SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')"
)
run_sql(
    f"ALTER TABLE {CHUNKS_TABLE_FQN} "
    f"SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
)
```

</details>

**Wallet check.** Still around one cent total for the session. Task 4 was three metadata writes against the warehouse. The lab's dollar spend was all in Task 3's embedding calls. Tesseract OCR (the slowest single step) cost zero dollars because Free Edition swallows the serverless compute against the daily quota.

## Cleanup

The cell below drops everything in reverse-creation order: chunks table, raw_docs table, volume contents, volume, schema (with CASCADE). Lab 5 recreates the chunks table from a Lab 4 re-run if you come back later. Re-running this cell is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order using the inline Databricks adapter. No critical (hourly-billed)
# resources in this lab so the canonical summary always reports zero critical
# destructions.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Databricks workspace may still hold lab objects. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $0.01 to $0.02.** Foundation Model API embedding calls on roughly 30 to 50 chunks land at one or two cents total. Tesseract OCR ran on serverless compute against the daily DBU quota (free). Metastore writes were free. The chunks table, raw_docs table, volume, and schema were destroyed by the cleanup cell above, so nothing is still running. Check your workspace billing console in 24 hours to confirm.

## Reflection

These are not graded. They are for you.

1. Walk through why `pypdf` is the wrong tool for a scanned-image PDF and the right tool for a text PDF. What is `pypdf` actually looking at when it parses, and what does `pytesseract` do that `pypdf` does not? When would you reach for `unstructured[pdf,image]` as a single-tool solution and what does that buy you over the per-format approach you used here?

2. The chunks table uses `chunk_id STRING` (a UUID) as the row identifier rather than `chunk_id BIGINT` with a monotonic counter. What breaks when you re-ingest the same document later (incremental refresh, not full rebuild) if you used integer IDs? How does the Vector Search Delta-Sync index downstream in Lab 5 react to row identity changes?

## Exam-Style Practice

**Question 1 (Domain 2, package selection per the official sample-question pattern):**

A GenAI Engineer is building a RAG application that relies on context retrieved from source documents that have been scanned and saved as image files in formats like .jpeg or .png. They want to develop a solution using the least amount of lines of code. Which Python package should be used to extract the text from the source documents?

A. beautifulsoup.
B. scrapy.
C. pytesseract.
D. pyquery.

<details><summary>Show answer</summary>

**Correct: C.**

- A is wrong: beautifulsoup parses HTML and XML, not images.
- B is wrong: scrapy is a web crawler; it does not OCR images.
- C is correct: pytesseract is a Python binding to the Tesseract OCR engine, the standard "minimal lines of code" path for extracting text from image files.
- D is wrong: pyquery is a jQuery-style HTML parser, also not an OCR tool.

This is verbatim the Question 3 pattern from the official sample-question set in the March 2026 exam guide.

</details>

**Question 2 (Domain 4, Delta chunks table design for Vector Search Delta-Sync):**

A GenAI engineer wants their chunks Delta table to feed a Mosaic AI Vector Search Delta-Sync index so the index picks up new chunks automatically as the table grows. Which table property is REQUIRED on the source table?

A. `delta.appendOnly = true`.
B. `delta.autoOptimize.autoCompact = true`.
C. `delta.enableChangeDataFeed = true`.
D. `delta.deletedFileRetentionDuration = '7 days'`.

<details><summary>Show answer</summary>

**Correct: C.**

- A is wrong: append-only is a write constraint, not the Vector Search prerequisite. Delta-Sync indexes do not require it.
- B is wrong: auto-compaction is a performance optimization, not the prerequisite.
- C is correct: Delta-Sync indexes read from the Change Data Feed; the source table must have CDF enabled. Without this, the Vector Search index has no signal to pick up new rows.
- D is wrong: file retention controls vacuum behavior, not Vector Search sync.

</details>